# Computer vision — visible mould screen

No public **Indian cattle-feed bag** image CSV exists (GrainSet-tiny is 695 MB of
single cereal kernels and is the wrong product). The laptop app deploys a **CNN**
trained in `train_mould_cnn.py` on synthetic 32×32 cake / silage / mould patches.
This notebook shows the farmer flag and the HSV side-check (HSV false-flags
uniform green fodder).

It does **not** output aflatoxin µg/kg or crude protein.


In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path("../..").resolve()
if not (ROOT / "smartfeed").exists():
    ROOT = Path(".").resolve()
    while ROOT != ROOT.parent and not (ROOT / "smartfeed").exists():
        ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data" / "processed"
print("ROOT", ROOT)
print("DATA", DATA)


ROOT /Users/sayangarai/Documents/Programming/Projects/SIH_PS_260111
DATA /Users/sayangarai/Documents/Programming/Projects/SIH_PS_260111/data/processed


In [2]:
import numpy as np
from PIL import Image
from smartfeed.cv_mould import assess_image, _hsv_mould_score

print(assess_image(visible_mould=False))
print(assess_image(visible_mould=True))

# Synthetic clean tan vs green-mould patches so the notebook runs without a 700 MB zip.
clean = np.full((80, 80, 3), (180, 150, 90), dtype=np.uint8)
mouldy = clean.copy()
mouldy[20:60, 20:60] = (40, 140, 50)
print("clean HSV", _hsv_mould_score(clean))
print("mouldy HSV", _hsv_mould_score(mouldy))

art = Path("artifacts")
art.mkdir(exist_ok=True)
Image.fromarray(mouldy).save(art / "demo_mould_patch.png")
print(assess_image(str(art / "demo_mould_patch.png")))


{'present': True, 'method': 'farmer_flag', 'mould_score': 0.0, 'flag': False, 'note': 'Farmer-reported visible mould. Do not convert this to ppb.'}
{'present': True, 'method': 'farmer_flag', 'mould_score': 1.0, 'flag': True, 'note': 'Farmer-reported visible mould. Do not convert this to ppb.'}
clean HSV {'green_frac': 0.0, 'dark_frac': 0.0, 'white_frac': 0.0, 'mould_score': 0.0, 'flag': False}
mouldy HSV {'green_frac': 0.25, 'dark_frac': 0.0, 'white_frac': 0.0, 'mould_score': 0.55, 'flag': True}
{'present': True, 'method': 'cnn_over_hsv', 'image': 'artifacts/demo_mould_patch.png', 'cnn_mould_proba': 1.0, 'hsv': {'green_frac': 0.25, 'dark_frac': 0.0, 'white_frac': 0.0, 'mould_score': 0.55, 'flag': True}, 'mould_score': 1.0, 'flag': True, 'note': 'cnn on 32x32 (trained on synthetic cake/silage/mould, not Indian bags). Not AFB1 µg/kg. HSV kept as a side check; it false-flags uniform green fodder.'}


In [3]:
art = Path("artifacts")
art.mkdir(exist_ok=True)
print("CNN metrics live in artifacts/metrics.json (written by train_mould_cnn.py). HSV is only a side-check.")


CNN metrics live in artifacts/metrics.json (written by train_mould_cnn.py). HSV is only a side-check.
